In [1]:
#import mortality rate of under 5, for all country
#Data Preparation
!pip install pycountry
import pandas as pd
import pycountry
import numpy as np

url = "https://ourworldindata.org/grapher/under-5-mortality-rate-sdgs.csv?v=1&csvType=full&useColumnShortNames=true"
mortality_under_5 = pd.read_csv(url, storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})
print(mortality_under_5.head())

#The mortality rate was collected in per 100, the rate was coverted to per 1000 by multipying per 100 to work with the general recorded rate of per 1000
mortality_under_5["per_1000"] = mortality_under_5["observation_value__indicator_child_mortality_rate__sex_total__wealth_quintile_total__unit_of_measure_deaths_per_100_live_births"] * 10
mortality_under_5.head()

#The name of countries and their ISO code in existence
countries = []
for c in pycountry.countries:
    countries.append({
        'country_name': c.name,
        'iso_alpha3': c.alpha_3
    })

iso_data = pd.DataFrame(countries).sort_values('country_name').reset_index(drop=True)
print(iso_data.head())

#Rename the column that contains the code for each country of iso data, so it can be merge with under_5 mortality data
iso_data = iso_data.rename(columns={'iso_alpha3': 'code', "country_name":"entity"})
iso_data.head()

#Merge under_5_mertality data on the left with iso_data so as to filter the data to all country,
#because the under_5_mortality data as some world back indicator in between which does not belong to any country
mortality_under_5_new = pd.merge(mortality_under_5, iso_data, on=['code', "entity"], how='left')
mortality_under_5_new.head()

#Validating my new data
print(mortality_under_5.shape)
print(mortality_under_5_new.shape)

#Then merge isn't the proper method, i will try filter method
mortality_under_5_new_2 = mortality_under_5[mortality_under_5["code"].isin(iso_data["code"])]
print(mortality_under_5_new_2.shape)
mortality_under_5_new_2.head()

#validating if we have all countries in the list
print(mortality_under_5_new_2["code"].unique())
unique_coutries = mortality_under_5_new_2["code"].unique()
print(len(unique_coutries))

#Importing data for income category for each country
import requests
import pandas as pd

# Pull country metadata including income classification
url = "https://api.worldbank.org/v2/country?format=json&per_page=300"
response = requests.get(url)
data = response.json()[1]
# index 0 is pagination info, index 1 is the actual data

# Flatten the nested structure
income_level = pd.DataFrame([
    {
        "Code": c["id"],
        "country_name": c["name"],
        "income_group": c["incomeLevel"]["value"],
        "region": c["region"]["value"]
    }
    for c in data
])

# Drop non-country aggregates (World Bank includes regional groupings here too)
income_level = income_level[income_level["income_group"] != "Aggregates"]

print(income_level["income_group"].unique())
income_level.head()

#Now i have my income level data, so i will like extract two columns from it to the mortarlity data
mortality_under_5_new_2["income_level"] = mortality_under_5_new_2[mortality_under_5_new_2['code'].isin(income_level['Code'])]['code'].map(income_level.set_index('Code')['income_group'])
mortality_under_5_new_2["regions"] = mortality_under_5_new_2[mortality_under_5_new_2['code'].isin(income_level['Code'])]['code'].map(income_level.set_index('Code')['region'])
print(mortality_under_5_new_2.head())
print(mortality_under_5_new_2["income_level"].unique())
print(mortality_under_5_new_2["regions"].unique())


#I waant to drop missing value in the data
mort_und_5_new_2_dropna = mortality_under_5_new_2.dropna()
print(mort_und_5_new_2_dropna.shape)
print(mort_und_5_new_2_dropna.isna().sum())

#Since my interest is working on years from year 2000 to later year.
mort_und_5_new_2_dropna_2000 = mort_und_5_new_2_dropna[mort_und_5_new_2_dropna["year"] >= 2000]
print(mort_und_5_new_2_dropna_2000.shape)

#validating which countries have dropped since i applied dropping missing valuea nd also
#applied filtered for years above 2000
print(mort_und_5_new_2_dropna_2000["code"].unique())
unique_coutries_2 = mort_und_5_new_2_dropna_2000["code"].unique()
print(len(unique_coutries_2))

# Rename data
mortality_under_5 = mort_und_5_new_2_dropna_2000
print(mortality_under_5.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 35.7 MB/s eta 0:00:00
        entity code  year  \
0  Afghanistan  AFG  1957   
1  Afghanistan  AFG  1958   
2  Afghanistan  AFG  1959   
3  Afghanistan  AFG  1960   
4  Afghanistan  AFG  1961   

   observation_value__indicator_child_mortality_rate__sex_total__wealth_quintile_total__unit_of_measure_deaths_per_100_live_births  
0                                          36.819660                                                                                
1                                          36.206852                                                                                
2                                          35.644524                                                                                
3                                          35.055080                                                                                
4                                          34.519325                               

/tmp/ipykernel_965/2903917864.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mortality_under_5_new_2["income_level"] = mortality_under_5_new_2[mortality_under_5_new_2['code'].isin(income_level['Code'])]['code'].map(income_level.set_index('Code')['income_group'])
/tmp/ipykernel_965/2903917864.py:79: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mortality_under_5_new_2["regions"] = mortality_under_5_new_2[mortality_under_5_new_2['code'].isin(income_level['Code'])]['code'].map(income_level.set_index('Cod

In [2]:
#Get general continent data which is different from the regional classification of world bank
!pip install pycountry_convert -q

import pycountry_convert as pc

def get_continent(iso_alpha3):
    try:
        iso_alpha2 = pc.country_alpha3_to_country_alpha2(iso_alpha3)
        continent_code = pc.country_alpha2_to_continent_code(iso_alpha2)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except:
        return None  # some small territories/regions won't map cleanly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 257.1/257.1 kB 8.5 MB/s eta 0:00:00


In [3]:
mortality_under_5['continent'] = mortality_under_5['code'].apply(get_continent)

# Check how many failed to map
print(mortality_under_5['continent'].isna().sum())
print(mortality_under_5[mortality_under_5['continent'].isna()]['entity'].tolist())
print(mortality_under_5["continent"].unique())

25
['East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor', 'East Timor']
['Asia' 'Europe' 'Africa' 'North America' 'South America' 'Oceania' None]


/tmp/ipykernel_965/1685249024.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mortality_under_5['continent'] = mortality_under_5['code'].apply(get_continent)


In [4]:
import plotly.express as px

latest = mortality_under_5[mortality_under_5['year'] == mortality_under_5['year'].max()]

fig = px.choropleth(
    latest,
    locations='code',
    color='per_1000',
    hover_name='entity',
    hover_data= 'continent',
    color_continuous_scale='Reds',
    labels={'under5_mortality_rate': 'Deaths per 1,000 live births'},
    title=f'Under-5 Mortality Rate by Country ({int(latest["year"].max())})'
)

fig.update_layout(
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(title='U5MR')
)

fig.show()

In [ ]:
# Requires:
!pip install kaleido==0.2.1

In [ ]:
#save the html format of the map
#fig.write_html(f'{'/content/drive/MyDrive/Portfolio/30 Days Challenge/Day 5'}/under5_mortality_choropleth_continent.html')

In [ ]:
#save the image format of the map
#fig.write_image(f'{'/content/drive/MyDrive/Portfolio/30 Days Challenge/Day 5'}/under5_mortality_choropleth.png', width=1400, height=800, scale=2)